# Predictive text and text generation

By [Allison Parrish](http://www.decontextualize.com/)

This notebook is a whirlwind tour of how certain kinds of predictive text generation work! By "predictive text generation" what I mean is any text generation method that is based around a statistical model that, given a certain stretch of text, "predicts" which bit of text should come next, based on probabilities learned from an existing corpus of text.

The code is written in Python, but you don't really need to know Python in order to use the notebook. Everything's pre-written for you, so you can just execute the cells, making small changes to the code as needed.

## Working with text files

Before we get started, we'll first need some text! Grab two [plain text files from Project Gutenberg](http://www.gutenberg.org/) (or from another source of your choice) and save them to the same directory as this notebook. (I suggest working with two files because we'll be running some code explicitly to "compare" two texts. Also, I think seeing two different outputs from the text generation methods discussed in this notebook will help you better understand how those methods work.) The code in the following cell loads into Python variables the contents of *two plain text files*, assigned to variables `text_a` and `text_b`. You'll need to replace the filenames with the names of the files that you downloaded, keeping the quotation marks (`"`) intact.

In [1]:
text_a = open("1342-0.txt").read()
text_b = open("84-0.txt").read()

These variables are *strings*, which are essentially just long lists of the characters that occur in the text, in the order that they occur. The code in the following cell shows the first two hundred characters of text A:

In [2]:
print(text_a[:200])

﻿The Project Gutenberg EBook of Pride and Prejudice, by Jane Austen

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away 


You can change `text_a` to `text_b` to see the output from your second text, or change `200` to a number of your choosing.

The `random.sample()` function gives us a random sampling of the contents of a variable (as long as that variable is a sequence of things, like a string or a list). So, for example, to see twenty random characters from text B:

In [4]:
import random
random.sample(text_b, 20)

['i',
 'o',
 'r',
 'a',
 'r',
 'd',
 't',
 'h',
 'r',
 'o',
 ' ',
 ' ',
 ' ',
 ' ',
 'c',
 'u',
 'u',
 'a',
 'e',
 ' ']

This isn't incredibly helpful on its own, but you'll notice that the characters it drew (probably) more or less follow the expected letter distribution for English (i.e., lots of `e`s and `n`s and `t`s).

Perhaps more interesting would be to see a randomly-sampled list of *words*. To do this, we'll make separate variables for the words in the text, using a Python function called `.split()`, which takes a string and turns it into a list of words contained in that string. The following cell makes two new variables that contain the words from both texts respectively:

In [5]:
a_words = text_a.split()
b_words = text_b.split()

Now, ten random words from both text A and text B:

In [6]:
random.sample(a_words, 10)

['Darcy',
 'Sir',
 'silence',
 'he',
 'consolation',
 'lady',
 'suit',
 'power',
 'family',
 'discovered']

In [7]:
random.sample(b_words, 10)

['my',
 'of',
 'of',
 'and',
 'the',
 'minutes',
 'that',
 'tempest,',
 'fixed',
 'terms']

The code in the following cell uses Python's `Counter` object to count the *most common* letters in the first of these texts:

In [9]:
from collections import Counter
Counter(text_a).most_common(12)

[(' ', 113941),
 ('e', 70344),
 ('t', 47283),
 ('a', 42156),
 ('o', 41138),
 ('n', 38430),
 ('i', 36273),
 ('h', 33883),
 ('r', 33293),
 ('s', 33292),
 ('d', 22247),
 ('l', 21282)]

Specifying the `a_words` variable gives the most frequent *words* instead:

In [10]:
Counter(a_words).most_common(12)

[('the', 4205),
 ('to', 4121),
 ('of', 3662),
 ('and', 3309),
 ('a', 1945),
 ('her', 1858),
 ('in', 1813),
 ('was', 1795),
 ('I', 1740),
 ('that', 1419),
 ('not', 1356),
 ('she', 1306)]

Compare these to the most common words in text B:

In [11]:
Counter(b_words).most_common(12)

[('the', 4056),
 ('and', 2971),
 ('of', 2741),
 ('I', 2719),
 ('to', 2142),
 ('my', 1631),
 ('a', 1394),
 ('in', 1125),
 ('was', 993),
 ('that', 987),
 ('with', 700),
 ('had', 679)]

What we're comparing here is called *unigram frequency*. ("Unigram" means a sequence of length one—more on this below.) For most English texts, the most frequent words in any given text will correspond closely to the most common 

## Frequency significance

One thing you can do with unigram frequencies is figure out which words are *most distinctive* in a given text. This would allow you to do things like [extract keywords from a text](https://github.com/aparrish/rwet/blob/master/quick-and-dirty-keywords.ipynb), or make a simple classifier that could tell you, for some stretch of text, whether that text is more similar to a given text A or text B. The code in the cell below implements a simple "distinctiveness" check. If you give it the `Counter` object from the code above, it will give you a list of words whose frequency in text B is most distinctive as compared to text A.

Don't worry about understanding this code—just execute it to make the functions available and then move on to the next cell. (For the curious, the code uses a [G-test](https://en.wikipedia.org/wiki/G-test) to determine significance.)

In [14]:
from scipy.stats import chi2_contingency
def compare_counts(a_count, b_count, count_threshold=1):
    a_total = sum(a_count.values())
    b_total = sum(b_count.values())
    sigs = []
    for k, v in a_count.items():
        if v <= count_threshold:
            continue
        sigs.append(
            (k,
             chi2_contingency([[v, a_total], [b_count[k], b_total]], lambda_=0)[0],
             "a" if v > b_count[k] else "b"))
    sigs.sort(key=lambda x: x[1], reverse=True)
    return sigs
def count_report(a_count, b_count, n=10, glue=""):
    compared = compare_counts(a_count, b_count)
    print("most significant")
    print("----------------")
    for item in compared[:n]:
        print(glue.join(item[0]), " (text ", item[2], ")", sep="")
    print()
    print("least significant")
    print("-----------------")
    for item in compared[-n:]:
        print(glue.join(item[0]), " (text ", item[2], ")", sep="")

Running the cell below will now give you the list of the most significant and least significant words in terms of their frequency in the given texts. The "most significant" words are the words that are most distinctive to either text. The "least significant" are words that more or less occur in the same frequency in both texts, and which therefore can't be considered distinctive.

In [15]:
count_report(Counter(a_words), Counter(b_words), 15)

most significant
----------------
my (text b)
I (text b)
Mr. (text a)
her (text a)
she (text a)
the (text a)
Mrs. (text a)
me (text b)
Miss (text a)
Darcy (text a)
and (text a)
Elizabeth (text a)
though (text a)
be (text a)
very (text a)

least significant
-----------------
sentiment (text a)
away (text a)
care (text a)
once (text a)
In (text a)
At (text a)
should (text a)
information (text a)
All (text a)
open (text a)
stood (text a)
society (text a)
you, (text a)
understand (text a)
sat (text a)


## N-grams

Unigrams are great, but there's only so much they can tell us about a text, since a unigram-based analysis throws out all information about *the order* in which words occur. So the first kind of text analysis that we’ll look at today is an n-gram model. An n-gram is simply a sequence of units drawn from a longer sequence; in the case of text, the unit in question is usually a character or a word. For convenience, we'll call the unit of the n-gram its *level*; the length of the n-gram is called its *order*. For example, the following is a list of all unique character-level order-2 n-grams in the word `condescendences`:

    co
    on
    nd
    de
    es
    sc
    ce
    en
    nc

And the following is an excerpt from the list of all unique word-level order-5 n-grams in *The Road Not Taken*:

    Two roads diverged in a
    roads diverged in a yellow
    diverged in a yellow wood,
    in a yellow wood, And
    a yellow wood, And sorry
    yellow wood, And sorry I

N-grams are used frequently in natural language processing and are a basic tool text analysis. Their applications range from programs that correct spelling to creative visualizations to compression algorithms to stylometrics to generative text. They can be used as the basis of a Markov chain algorithm—and, in fact, that’s one of the applications we’ll be using them for later in this lesson.

### Finding and counting word pairs

So how would we go about writing Python code to find n-grams? We'll start with a simple task: finding *word pairs* in a text. A word pair is essentially a word-level order-2 n-gram; once we have code to find word pairs, we’ll generalize it to handle n-grams of any order.



In [16]:
def ngrams_for_sequence(n, seq):
    return [tuple(seq[i:i+n]) for i in range(len(seq)-n+1)]

Using this function, here are random character-level n-grams of order 9 from text A:

In [17]:
import random
a_9grams = ngrams_for_sequence(9, text_a)
random.sample(a_9grams, 10)

[(' ', 'C', 'o', 'l', 'l', 'i', 'n', 's', ' '),
 ('v', 'e', '.', ' ', 'T', 'h', 'e', ' ', 'w'),
 ('t', '\n', 'w', 'a', 's', ' ', 'a', 'n', ' '),
 ('t', ',', ' ', 'u', 'n', 'a', 'b', 'l', 'e'),
 ('v', 'e', 'n', 't', ' ', 'h', 'i', 's', ' '),
 ('l', ' ', 'l', 'i', 'g', 'h', 't', '\n', 'm'),
 (' ', 'h', 'e', 'r', ' ', 'a', 's', '\n', 'h'),
 ('b', 'e', 'l', 'i', 'e', 'v', 'e', ' ', 'm'),
 ('e', ' ', 'w', 'h', 'i', 'c', 'h', ' ', 's'),
 ('c', 'o', 'm', 'p', 'r', 'e', 'h', 'e', 'n')]

Or all the word-level 5-grams from text B:

In [18]:
b_word_5grams = ngrams_for_sequence(5, text_b.split())
random.sample(b_word_5grams, 10)

[('to', 'state', 'those', 'facts', 'which'),
 ('for', 'my', 'letters', 'with', 'feverish'),
 ('me', 'inexpressible', 'pleasure.', 'But', 'a'),
 ('upright', 'in', 'it.', 'No', 'wood,'),
 ('the', 'pursuit', 'of', 'knowledge', 'is'),
 ('presented', 'a', 'thousand', 'objects', 'that'),
 ('become', 'acquainted', 'with', 'any', 'of'),
 ('and', 'their', 'idol,', 'and', 'something'),
 ('still', 'proceed', 'over', 'the', 'untamed'),
 ('be', 'friendless', 'is', 'indeed', 'to')]

All of the bigrams (ngrams of order 2) from the string `condescendences`:

In [19]:
ngrams_for_sequence(2, "condescendences")

[('c', 'o'),
 ('o', 'n'),
 ('n', 'd'),
 ('d', 'e'),
 ('e', 's'),
 ('s', 'c'),
 ('c', 'e'),
 ('e', 'n'),
 ('n', 'd'),
 ('d', 'e'),
 ('e', 'n'),
 ('n', 'c'),
 ('c', 'e'),
 ('e', 's')]

And of course we can use it in conjunction with a `Counter` to find the most common n-grams in a text:

In [21]:
from collections import Counter
a_count = Counter(ngrams_for_sequence(3, text_a.split()))
b_count = Counter(ngrams_for_sequence(3, text_b.split()))

The most common 3-grams from text A:

In [23]:
a_count.most_common(12)

[(('as', 'soon', 'as'), 45),
 (('she', 'could', 'not'), 42),
 (('I', 'do', 'not'), 39),
 (('that', 'he', 'had'), 35),
 (('I', 'am', 'sure'), 35),
 (('could', 'not', 'be'), 30),
 (('it', 'would', 'be'), 28),
 (('as', 'well', 'as'), 27),
 (('by', 'no', 'means'), 26),
 (('would', 'have', 'been'), 26),
 (('that', 'it', 'was'), 25),
 (('one', 'of', 'the'), 25)]

... and the most common 3-grams from text B:

In [24]:
b_count.most_common(12)

[(('which', 'I', 'had'), 38),
 (('I', 'could', 'not'), 31),
 (('I', 'did', 'not'), 31),
 (('that', 'I', 'had'), 25),
 (('that', 'I', 'was'), 22),
 (('me,', 'and', 'I'), 19),
 (('that', 'I', 'might'), 18),
 (('Project', 'Gutenberg-tm', 'electronic'), 18),
 (('that', 'I', 'should'), 17),
 (('I', 'do', 'not'), 16),
 (('the', 'Project', 'Gutenberg'), 15),
 (('one', 'of', 'the'), 15)]

The same statistical significance test we used on unigrams can also give us the most distinctive 3-grams:

In [25]:
count_report(a_count, b_count, 12, " ")

most significant
----------------
she could not (text a)
I am sure (text a)
me, and I (text b)
as soon as (text a)
I could not (text b)
Mrs. Bennet was (text a)
by no means (text a)
could not be (text a)
Mr. and Mrs. (text a)
that I had (text b)
that I should (text b)
that she had (text a)

least significant
-----------------
when they were (text a)
if you are (text a)
soon as I (text a)
it is the (text a)
of the day (text a)
the particulars of (text a)
the loss of (text a)
a mixture of (text a)
it was to (text a)
but there was (text a)
was able to (text a)
But it was (text a)


## Markov models: what comes next?

Now that we have the ability to find and record the n-grams in a text, it’s time to take our analysis one step further. The next question we’re going to try to answer is this: Given a particular n-gram in a text, what is most likely to come next?

We can imagine the kind of algorithm we’ll need to extract this information from the text. It will look very similar to the code to find n-grams above, but it will need to keep track not just of the n-grams but also a list of all units (word, character, whatever) that *follow* those n-grams.

Let’s do a quick example by hand. This is the same character-level order-2 n-gram analysis of the (very brief) text “condescendences” as above, but this time keeping track of all characters that follow each n-gram:

| n-grams |	next? |
| ------- | ----- |
|co| n|
|on| d|
|nd| e, e|
|de| s, n|
|es| c, (end of text)|
|sc| e|
|ce| n, s|
|en| d, c|
|nc| e|

From this table, we can determine that while the n-gram `co` is followed by n 100% of the time, and while the n-gram `on` is followed by `d` 100% of the time, the n-gram `de` is followed by `s` 50% of the time, and `n` the rest of the time. Likewise, the n-gram `es` is followed by `c` 50% of the time, and followed by the end of the text the other 50% of the time.

Exercise: Imagine (or even better, write out) what this table might look like if you were analyzing words instead of characters, with a source text of your choice.

### Markov chains: Generating text from a Markov model

The Markov models we created above don't just give us interesting statistical probabilities. It also allows us generate a *new* text with those probabilities by *chaining together predictions*. Here’s how we’ll do it, starting with the order 2 character-level Markov model of `condescendences`: (1) start with the initial n-gram (`co`)—those are the first two characters of our output. (2) Now, look at the last *n* characters of output, where *n* is the order of the n-grams in our table, and find those characters in the “n-grams” column. (3) Choose randomly among the possibilities in the corresponding “next” column, and append that letter to the output. (Sometimes, as with `co`, there’s only one possibility). (4) If you chose “end of text,” then the algorithm is over. Otherwise, repeat the process starting with (2). Here’s a record of the algorithm in action:

    co
    con
    cond
    conde
    conden
    condend
    condendes
    condendesc
    condendesce
    condendesces
    
As you can see, we’ve come up with a word that looks like the original word, and could even be passed off as a genuine English word (if you squint at it). From a statistical standpoint, the output of our algorithm is nearly indistinguishable from the input. This kind of algorithm—moving from one state to the next, according to a list of probabilities—is known as a Markov chain generator.

### Generating with Markovify

Fortunately, with the invention of digital computers, you don't have to perform this algorithm by hand! In fact, Markov chain text generation has been a pastime of poets and programmers going back [all the way to 1983](https://www.jstor.org/stable/24969024), so it should be no surprise that there are many implementations of the idea in Python that you can download and install. The one we're going to use is [Markovify](https://github.com/jsvine/markovify), a Markov chain text generation library originally developed for BuzzFeed, apparently. It comes with a lot of extra niceties that will make our lives easier, but underneath the hood, it implements an algorithm very similar to the one we just did by hand above.

To install Markovify on your computer, run the cell below:

In [26]:
!pip install markovify

You are using pip version 9.0.3, however version 10.0.1 is available.
You should consider upgrading via the 'pip install --upgrade pip' command.


And then run this cell to make the library available in your notebook:

In [27]:
import markovify

The code in the following cell creates a new text generator, using the text in the variable specified to build the Markov model, which is then assigned to the variable `generator_a`.

In [28]:
generator_a = markovify.Text(text_a)

You can then call the `.make_sentence()` method to generate a sentence from the model:

In [29]:
print(generator_a.make_sentence())

Mr. Darcy could not have happened; but poor consolation to think our friend mercenary.”


The `.make_short_sentence()` method allows you to specify a maximum length for the generated sentence:

In [33]:
print(generator_a.make_short_sentence(50))

I cannot consider your daughters.


By default, Markovify tries to generate a sentence that is significantly different from any existing sentence in the input text. As a consequence, sometimes the `.make_sentence()` or `.make_short_sentence()` methods will return `None`, which means that in ten tries it wasn't able to generate such a sentence. You can work around this by increasing the number of times it tries to generate a sufficiently unique sentence using the `tries` parameter:

In [34]:
print(generator_a.make_short_sentence(40, tries=100))

“But why should _we_?”


Or by disabling the check altogether with `test_output=False`:

In [35]:
print(generator_a.make_short_sentence(40, test_output=False))

It could not mention before me.”


### Changing the order

When you create the model, you can specify the order of the model using the `state_size` parameter. It defaults to 2. Let's make two model with different orders and compare:

In [36]:
gen_a_1 = markovify.Text(text_a, state_size=1)
gen_a_4 = markovify.Text(text_a, state_size=4)

In [41]:
print("order 1")
print(gen_a_1.make_sentence(test_output=False))
print()
print("order 4")
print(gen_a_4.make_sentence(test_output=False))

order 1
“What a countenance, such an explanation of each with a ring at least it would scarcely needed an attachment.

order 4
She wrote cheerfully, seemed surrounded with comforts, and mentioned nothing which she could not have formed a very pleasing opinion of conjugal felicity or domestic comfort.


In general, the higher the order, the more the sentences will seem "coherent" (i.e., more closely resembling the source text). Lower order models will produce more variation. Deciding on the order is usually a matter of taste and trial-and-error.

### Changing the level

Markovify, by default, works with *words* as the individual unit. It doesn't come out-of-the-box with support for character-level models. The following code defines a new kind of Markovify generator that implements character-level models. Execute it before continuing:

In [42]:
class SentencesByChar(markovify.Text):
    def word_split(self, sentence):
        return list(sentence)
    def word_join(self, words):
        return "".join(words)

Any of the parameters you passed to `markovify.Text` you can also pass to `SentencesByChar`. The `state_size` parameter still controls the order of the model, but now the n-grams are characters, not words.

The following cell implements a character-level Markov text generator for the word "condescendences":

In [43]:
con_model = SentencesByChar("condescendences", state_size=2)

Execute the cell below to see the output—it'll be a lot like what we implemented by hand earlier!

In [44]:
con_model.make_sentence()

'condencescencesces'

Of course, you can use a character-level model on any text of your choice. So, for example, the following cell creates a character-level order-7 Markov chain text generator from text A:

In [45]:
gen_a_char = SentencesByChar(text_a, state_size=7)

And the cell below prints out a random sentence from this generator. (The `.replace()` is to get rid of any newline characters in the output.)

In [46]:
print(gen_a_char.make_sentence(test_output=False).replace("\n", " "))

There, Mrs. Bennet one day.


### Combining models

Markovify has a handy feature that allows you to *combine* models, creating a new model that draws on probabilities from both of the source models. You can use this to create hybrid output that mixes the style and content of two (or more!) different source texts. To do this, you need to create the models independently, and then call `.combine()` to combine them.

In [47]:
generator_a = markovify.Text(text_a)
generator_b = markovify.Text(text_b)
combo = markovify.combine([generator_a, generator_b], [0.5, 0.5])

The bit of code `[0.5, 0.5]` controls the "weights" of the models, i.e., how much to emphasize the probabilities of any model. You can change this to suit your tastes. (E.g., if you want mostly text A with but a *soupçon* of text B, you would write `[0.9, 0.1]`. Try it!) 

Then you can create sentences using the combined model:

In [48]:
print(combo.make_sentence())

The hall, the dining-room, which fronted the lane, and were assisted in their native country.


### Bringing it all together

I've pre-written some code below to make it easy for you to experiment and produce output from Markovify. Just make adjustments to the values assigned to the variables in the cell below:

In [49]:
# change to "word" for a word-level model
level = "char"
# controls the length of the n-gram
order = 7
# controls the number of lines to output
output_n = 14
# weights between the models; text A first, text B second.
# if you want to completely exclude one model, set its corresponding value to 0
weights = [0.5, 0.5]
# limit sentence output to this number of characters
length_limit = 280

(The lines beginning with `#` are "comments"—they don't do anything, they're just there to explain what's happening in the code.)

After making your changes above, run the cell below to generate text according to your parameters. Repeat as necessary until you get something you really like!

In [50]:
model_cls = markovify.Text if level == "word" else SentencesByChar
gen_a = model_cls(text_a, state_size=order)
gen_b = model_cls(text_b, state_size=order)
gen_combo = markovify.combine([gen_a, gen_b], weights)
for i in range(output_n):
    out = gen_combo.make_short_sentence(length_limit, test_output=False)
    out = out.replace("\n", " ")
    print(out)
    print()

She resource, her minutes, you know, when my father to the convenience of natural hideous monsters that Lydia's for her too in its execution.”

“Oh, yes.

A man might lower, but followed myself a little; but I have more than his affectionate praise, and circumstance of his; she was “very glad to for the satisfied, though well-bred, were ever with the happiness.

And with a bow to Mr. Darcy called from what patriot fell.

I have still was all that he was.

I felt that the length the while he was reflection for her.

“After the felicity of his maker owe him not; call out of the morning without report; and there are so strange and trademark, my dear sister had been unfolded against him.

He absolutely settled upon it.

As the mockery and felt that _you_ are dancing with his daughters uncommonly well repaid my fair creation; I imagined; his concerning she been imposed.

I have describe and praises my weak and fourth with eagerly desired information, till her grateful to write such a number

## Neural network text prediction with `textgenrnn`

Like a [Markov chain](ngrams-and-markov-chains.ipynb), a recurrent neural network (RNN) is a way to make predictions about what will come next in a sequence. For our purposes, the sequence in question is a sequence of characters, and the prediction we want to make is *which character will come next*. Both Markov models and recurrent neural networks do this by using statistical properties of text to make a *probability distribution* for what character will come next, given some information about what comes before. The two procedures work very differently internally, and we're not going to go into the gory details about implementation here. (But if you're interested in the gory details, [here's a good place to start](https://karpathy.github.io/2015/05/21/rnn-effectiveness/).) For our purposes, the main *functional* difference between a Markov chain and a recurrent neural network is the *portion* of the sequence used to make the prediction. A Markov model uses a fixed window of history from the sequence, while an RNN (theoretically) uses the *entire history* of the sequence.

## Start with Markov

To illustrate, here's that Markov model of the word "condescendences." In a Markov model based on bigrams from this string of characters, you'd make a list of bigrams and the characters that follow those bigrams, like so:

| n-grams |	next? |
| ------- | ----- |
| co      | n |
| on      | d |
| nd      | e, e |
| de      | s, n |
| es      | c, (end of text) |
| sc      | e |
| ce      | n, s |
| en      | d, c |
| nc      | e |

You could also write this as a probability distribution, with one column for each bigram. The value in each cell indicates the probability that the character following the bigram in a given row will be followed by the character in a given column:

| n-grams | c | o | n | d | e | s | END |
| ------- | - | - | - | - | - | - | --- |
| co      | 0 | 0 | 1.0 | 0 | 0 | 0 | 0 | 
| on      | 0 | 0 | 0 | 1.0 | 0 | 0 | 0 | 
| nd      | 0 | 0 | 0 | 0 | 1.0 | 0 | 0 | 
| de      | 0 | 0 | 0.5 | 0 | 0 | 0.5 | 0 |
| es      | 0.5 | 0 | 0 | 0 | 0 | 0 | 0.5 |
| sc      | 0 | 0 | 0 | 0 | 1.0 | 0 | 0 |
| ce      | 0 | 0 | 0.5 | 0 | 0 | 0.5 | 0 |
| en      | 0.5 | 0 | 0 | 0.5 | 0 | 0 | 0 |
| nc      | 0 | 0 | 0 | 0 | 1.0 | 0 | 0 |

Each row of this table is a *probability distribution*, meaning that it shows how probable a given letter is to follow the n-gram in the original text. In a probability distribution, all of the values add up to 1.

Fitting a Markov model to the data is a matter of looking at each sequence of characters in a given text, and updating the table of probability distributions accordingly. To make a prediction from this table, you can "sample" from the probability distribution for a given n-gram (i.e., sampling from the distribution for the bigram `de`, you'd have a 50% chance of picking `n` and a 50% chance of picking `s`).

Another way of thinking about this Markov model is as a (hypothetical!) function *f* that takes a bigram as a parameter and returns a probability distribution for that bigram:

    f("ce") → [0.0, 0.0, 0.5, 0.0, 0.0, 0.5, 0.0]
    
(Note that the values at each index in this distribution line up with the columns in the table above.)
    
The items in the list returned from this function correspond to the probability for the corresponding next character, as given in the table. To sample from this list, you'd pick randomly among the indices according to their probabilities, and then look up the corresponding character by its position in the table.

To generate new text from this model:

1. Set your output string to a randomly selected n-gram
2. Sample a letter from the probability distribution associated with the n-gram at the end of the output string
3. Append the sampled letter to the end of the string
4. Repeat from (2) until the END token is reached

Of course, you don't write this function by hand! When you're creating a Markov model from your data (or "training" the model), you're essentially asking the computer to write this function *for you*. In this sense, a Markov model is a very simple kind of machine learning, since the computer "learns" the probability distribution from the data that you feed it.

## A (very) simplified explanation of RNNs

The mechanism by which a recurrent neural network "learns" probability distributions from sequences is much more sophisticated than the mechanism used in a Markov model, but functionally they're very similar: you give the computer some data to "train" on, and then ask it to automatically create a function that will return a probability distribution of what comes next, given some input. An RNN differs from a Markov chain in that to predict the next item in the sequence, you pass in *the entire sequence* instead of just the most recent n-gram.

In other words, you can (again, hypothetically) think of an RNN as a way of automatically creating a function *f* that takes a sequence of characters of arbitrary length and returns a probability distribution for which character comes next in the sequence. Unlike a Markov chain, it's possible to *improve* the accuracy of the probability distribution returned from this function by training on the same data multiple times.

Let's say that we want to train the RNN on the string "condescendences" to learn this function, and we want to make a prediction about what character is most likely to follow the sequence of characters "condescendence". When training a neural network, the process of learning a function like this works iteratively: you start off with a function that essentially gives a uniform probability distribution for each outcome (i.e., no one outcome is considered more likely than any other):

    f("condescendences") → [0.14, 0.14, 0.14, 0.14, 0.14, 0.14, 0.14] (after zero passes through the data)
    
... and as you iterate over the training data (in this case, the word "condescendences"), the probability distribution  gradually improves, ideally until it comes to accurately reflect the actual observed distribution (in the parlance, until it "converges"). After some number of passes through the data, you might expect the automatically-learned function to return distributions like this:

    f("condescendences") → [0.01, 0.02, 0.01, 0.03, 0.01, 0.9, 0.02] (after n passes through the data)

A single pass through the training data is called an "epoch." When it comes to any neural network, and RNNs in particular, more epochs is almost always better.

To generate text from this model:

1. Initialize your output string to an empty string, or a random character, or a starting "prefix" that you specify;
2. Sample the next letter from the distribution returned for the current output string;
3. Append that character to the end of the output string;
4. Repeat from (2)

Of course, in a real life application of both a Markov model and an RNN, you'd normally have more than seven items in the probability distribution! In fact, you'd have one element in the probability distribution for every possible character that occurs in the text. (Meaning that if there were 100 unique characters found in the text, the probability distribution would have 100 items in it.)

## Markov chains vs RNNs    

The primary benefit of an RNN over a Markov model for text generation is that an RNN takes into account *the entire history* of a sequence when generating the next character. This means that, for example, an RNN can theoretically learn how to close quotes and parentheses, which a Markov chain will never be able to reliably do (at least for pairs of quotes and parentheses longer than the n-gram of the Markov chain).

The drawback of RNNs is that they are *computationally expensive*, from both a processing and memory perspective. This is (again) a simplification, but internally, RNNs work by "squishing" information about the training data down into large matrices, and make predictions by performing calculations on these large matrices. That means that you need a lot of CPU and RAM to train an RNN, and the resulting models (when stored to disk) can be very large. Training an RNN also (usually) takes a lot of time.

Another consideration is the size of your corpus. Markov models will give interesting and useful results even for very small datasets, but RNNs require large amounts of data to train—the more data the better.

So what do you do if you *don't* have a very large corpus? Or if you don't have a lot of time to train on your corpus?

## RNN generation from pre-trained models

Fortunately for us, developer and data scientist [Max Woolf](https://github.com/minimaxir) has made a Python library called [textgenrnn](https://github.com/minimaxir/textgenrnn) that makes it really easy to experiment with RNN text generation. This library includes a model (according to the documentation) "trained on hundreds of thousands of text documents, from Reddit submissions (via BigQuery) and Facebook Pages (via my Facebook Page Post Scraper), from a very diverse variety of subreddits/Pages," and allows you to use this model as a starting point for your own training.

To install textgenrnn, you'll probably want to install [Keras](http://keras.io/) first. With Anaconda:

In [51]:
!conda install -y keras

Solving environment: done

# All requested packages already installed.



Then install textgenrnn with `pip`:

In [52]:
!pip install --upgrade textgenrnn

Requirement already up-to-date: textgenrnn in /Users/allison/anaconda/lib/python3.6/site-packages
Requirement already up-to-date: h5py in /Users/allison/anaconda/lib/python3.6/site-packages (from textgenrnn)
Requirement already up-to-date: scikit-learn in /Users/allison/anaconda/lib/python3.6/site-packages (from textgenrnn)
Requirement already up-to-date: keras>=2.1.5 in /Users/allison/anaconda/lib/python3.6/site-packages (from textgenrnn)
Requirement already up-to-date: numpy>=1.7 in /Users/allison/anaconda/lib/python3.6/site-packages (from h5py->textgenrnn)
Requirement already up-to-date: six in /Users/allison/anaconda/lib/python3.6/site-packages (from h5py->textgenrnn)
Requirement already up-to-date: keras-applications==1.0.2 in /Users/allison/anaconda/lib/python3.6/site-packages (from keras>=2.1.5->textgenrnn)
Requirement already up-to-date: scipy>=0.14 in /Users/allison/anaconda/lib/python3.6/site-packages (from keras>=2.1.5->textgenrnn)
Requirement already up-to-date: keras-prepr

Once it's installed, import the `textgenrnn` class from the package:

In [53]:
from textgenrnn import textgenrnn

Using TensorFlow backend.


And create a new `textgenrnn` object like so. (The `name` parameter controls the filename used when automatically saving the model to disk, so pick something descriptive!)

In [54]:
textgen = textgenrnn(name="text_a")

This object has a `.generate()` method which will, by default, generate text from the pre-trained model only.

In [56]:
textgen.generate()

The most of the story of sisters was a digital to be any adopting the first time and it was to wear one of the working on the side of the face of this picture of my games on its phone?



The `textgenrnn` library needs a data structure called a "list of strings" as its source text for training. We'll use Markovify's `split_into_sentences` method to turn our plain-text input files into lists of sentences like so:

In [57]:
from markovify.splitters import split_into_sentences
text_a_sentences = split_into_sentences(text_a)
text_b_sentences = split_into_sentences(text_b)

Here are five random sentences from both texts:

In [58]:
random.sample(text_a_sentences, 5)

['But your\n_family_ owe me nothing.',
 'Mr. Darcy bowed.',
 "Mrs. Reynolds informed them that it had been taken in his father's\nlifetime.",
 '“I should not be surprised,” said Darcy, “if he were to give it up as\nsoon as any eligible purchase offers.”',
 'I now give it to _you_, if you are resolved on\nhaving him.']

In [59]:
random.sample(text_b_sentences, 5)

['“He did not succeed.',
 'I found that the\nsparrow uttered none but harsh notes, whilst those of the blackbird and\nthrush were sweet and enticing.',
 'I trod\nheaven in my thoughts, now exulting in my powers, now burning with the idea\nof their effects.',
 'The old man, I could perceive,\noften endeavoured to encourage his children, as sometimes I found that\nhe called them, to cast off their melancholy.',
 'As he said this his countenance became expressive of a calm, settled\ngrief that touched me to the heart.']

To train a text generator on your own text, use the `.train_on_texts()` method, passing in a list of strings. The `num_epochs` parameter allows you to indicate how many epochs (i.e., passes over the data) should be performed. The more epochs the better, especially for shorter texts, but you'll get okay results even with just a few.

Training a neural network usually takes a really long time! So it makes sense to "try out" a text before committing to the many hours it might take to train the network on the full text. The following example trains the neural network on just the first 100 lines from text A, which lets you get an idea of what the output will look like when training on its entire contents. You'll notice that the `train_on_texts()` function prints output as it goes, showing what the generated text is likely to look like.

In [60]:
textgen.train_on_texts(text_a_sentences[:100], num_epochs=3)

Training on 8,345 character sequences.
Epoch 1/3
65/65 [==============================] - 39s 597ms/step - loss: 2.1617
####################
Temperature: 0.2
####################
“I the time of the competuting of the own dear make it internot the how to take the time of the fortne and wife one of the mary to make the own the such and the own the how one of the woman of the woman introduce of the notese of the many own wife internet intended one of the sourh, that have a ma

“I am anywhe that will be make it internet internet and will the notesself internet,” you have a compand of the wors of the worst to go one of the may to take the surright.”

“I am anywhan the may will be make the may or inter the fortnet internet, and make that the may was such and the how of the such and the own compenting the serville intended the time of the day of the woman of the woman of the woman of the notesisd of the mary of the competungare of the day of the

####################
Temperature: 0.5
########

After training, you can generate new text using the `.generate()` method again:

In [61]:
textgen.generate()

“What do you may go; that he may be mind it all of the surrounding and I am so sure that you him dead them as married; “sen a should but I conservent upon marrying the man and she will go before!



The results aren't very interesting because by default the generator is very conservative in how it samples from the probability distribution. You can use the `temperature` parameter to make the sampling a bit more likely to pick improbable outcomes. The higher the value, the weirder the results. The default is 0.2, and going above 1.0 is likely to produce unacceptably strange results:

In [62]:
textgen.generate(temperature=0.5)

Bennet, so you shoul not acquise her that a few that the same is will be must the such a man of offired in a base of the same, that I am so when I comes will be must be into a little his consider and the pacefomms delight it will be you as “singed to make a let inters.”



In [63]:
textgen.generate(temperature=0.9)

“What do you feeling you must all with!



In [64]:
textgen.generate(temperature=1.5)

“My pc; ifew nightit relengingfockedbed, af.



If you pass a number *n* to the `.generate()` method as its first parameter, `.generate()` will print out *n* instances of text generation from the model. The code in the following cell prints out ten examples from the specified temperature:

In [66]:
textgen.generate(10, temperature=0.5)

You may fought her for one of them, you must do not sure what a man detils that have the same; the them of my daughters, the mary in the asus, that in the hate of before?

“I do not for your surrung to make may be the time of me, Mr. Bennet, “the dear that so adjusting the four Mr. Bingley has seen the sort in the party.”

“It is not you be said _fine _.

“What can is see you must the most them?”

“But I am sure for my daughters.

“A shell has a new for my pic a surchabers.

“I am sure if no returned to have see you much to said her dear Jane to see the man of marrying that he comes in them on his wife what I am a part the littler of the new woman for her man introduce that would so for my daughters what I be for a surrey it, I mean thinking them.

“But the times indest and see you all little advised on the own bennet, or what she was been a mass as that it are you ampressed it affect to know and what a surrund her was a surround the visit, and the respost of a surrey to be more of my 

(This may take a little while.)

When you're satisfied with the results and you're ready to train on all of the sentences, just remove the `[:100]` from the call to `.train_on_texts()`:

In [ ]:
textgen.train_on_texts(text_a_sentences, num_epochs=5)

The textgenrnn library automatically saves the model to disk after each epoch in the same directory as this notebook. You can load a model you've previously trained by passing its filename to the `textgenrnn` function:

In [68]:
textgen = textgenrnn("text_a_weights.hdf5")

And then you can call the `.generate()` method as normal:

In [70]:
textgen.generate(10, temperature=0.5)

“You have the dear Mrs.

“I am fortune forthned or that I make it will be the nerves.”

“My fried in some news.

“Why did not such a marry in the introduce of the day.”

“Deleagly that it use to a few of my dear his monthord forthorress need and not may said you may have a mass her cried of the visit imponsire advise the store of them.

“But the compand of themselves have her theres her not one of them of her fining her daughters.

“It was see with you hear that you and make in her friend to some adsently to be will do not of her use of the nerves.

“I have cliefer that you have my consider forthorries that he should hear them to make it about them.”

“Derucate have a serving to taken the best forthorrowing him of Mr. Bingley was or it in sourh, that he sure that he has to so be for the empt and a man for your have my daughters when has no will like the daughters.

_Houghthere and will be you that was the them.



### Generating with shorter texts

I've found that `textgenrnn` works especially well with very short, word-length texts. For example, download [this file of human moods](https://github.com/dariusk/corpora/blob/master/data/humans/moods.json) from Corpora Project, and put it in the same directory as this notebook.

Then load the JSON file and grab just the list of words naming moods:

In [71]:
import json
mood_data = json.loads(open("./moods.json").read())
moods = mood_data['moods']

And create another textgenrnn object:

In [72]:
mood_gen = textgenrnn(name="moods")

Now, train the RNN on these moods. One epoch will do the trick:

In [73]:
mood_gen.train_on_texts(moods, num_epochs=1)

Training on 6,651 character sequences.
Epoch 1/1
51/51 [==============================] - 31s 607ms/step - loss: 2.2394
####################
Temperature: 0.2
####################
intatted

contrent

reserted

####################
Temperature: 0.5
####################
disnarned

piped

noest

####################
Temperature: 1.0
####################
shmedians

ermssty innockuistic

teguis



Now generate a list of new moods:

In [74]:
mood_gen.generate(25, temperature=0.5)

deantenic

resertly

accorruative

taped

asteriated

abled

distressed

expload

patenced

resuped

intranted

unbullet

disafted

adventy

enthely

discarned

dislansed

cured

contive

samiden

enforrems

leaged

justened

devellated

wristent



## Further reading

* [This notebook from the creator of textgenrnn](https://github.com/minimaxir/textgenrnn/blob/master/docs/textgenrnn-demo.ipynb) covers everything about the library that I covered in this tutorial—and much more, including how to start generation from a particular "seed" and how to save and load models (useful if you spent an afternoon training a model on your own corpus and don't want to have to do it again!)
* Take a look at [Janelle Shane's wonderful overview of how she uses RNNs in her process](http://aiweirdness.com/faq). And then take a look at her [wonderful creative work with RNNs](http://aiweirdness.com/).
* Hayes, Brian. “Computer recreations.” Scientific American, vol. 249, no. 5, 1983, pp. 18–31. JSTOR, http://www.jstor.org/stable/24969024. (Original column from Scientific American that described how Markov chain text generation works—very readable! I can send a PDF, hit me up.)
* [A Travesty Generator for Micros](https://elmcip.net/critical-writing/travesty-generator-micros) is a follow-up to Hayes' article that has some more theory and an actual Pascal listing (which is now mostly of only historical interest).
* [This notebook](https://github.com/aparrish/rwet/blob/master/ngrams-and-markov-chains.ipynb) shows how to implement a Markov chain generator from scratch in Python, if you're interested in such things!